# 📊 Préparation des données — Collections Inria HAL

Ce notebook récupère les publications des 3 collections HAL et génère les fichiers `publications.json` et `stats.json` pour chaque dashboard.

**Collections :**
- `PEPR_SANTENUM` — Santé Numérique
- `AGROECONUM` — Agroécologie et Numérique
- `INTERSTICES` — Sciences du numérique (vulgarisation)

In [ ]:
import requests
import json
import re
import os
from pathlib import Path

# Configuration des collections
COLLECTIONS = {
    'PEPR_SANTENUM': {
        'name': 'PEPR Santé Numérique',
        'output_dir': 'pepr_santenum/data'
    },
    'AGROECONUM': {
        'name': 'Agroécologie et Numérique',
        'output_dir': 'agroeconum/data'
    },
    'INTERSTICES': {
        'name': 'Interstices',
        'output_dir': 'interstices/data'
    }
}

HAL_BASE_URL = 'https://api.archives-ouvertes.fr/search/{collection}/'
HAL_FIELDS = ','.join([
    'halId_s', 'title_s', 'docType_s', 'publicationDateY_i',
    'authFullName_s', 'authLastName_s', 'authFirstName_s',
    'labStructName_s', 'labStructAddress_s', 'labStructCountry_s',
    'instStructName_s', 'instStructCountry_s',
    'domain_s', 'keyword_s', 'journalTitle_s',
    'language_s', 'openAccess_bool', 'doiId_s'
])
print('✅ Configuration chargée')

In [ ]:
# Dictionnaire de villes connues pour améliorer l'extraction
CITY_MAP = {
    'sophia antipolis': 'Sophia Antipolis',
    'grenoble': 'Grenoble',
    'paris': 'Paris',
    'rennes': 'Rennes',
    'bordeaux': 'Bordeaux',
    'talence': 'Talence',
    'palaiseau': 'Palaiseau',
    'saclay': 'Saclay',
    'rocquencourt': 'Rocquencourt',
    'nancy': 'Nancy',
    'lyon': 'Lyon',
    'lille': 'Lille',
    'montpellier': 'Montpellier',
    'toulouse': 'Toulouse',
    'strasbourg': 'Strasbourg',
    'nantes': 'Nantes',
    'nice': 'Nice',
    'marseille': 'Marseille',
    'versailles': 'Versailles',
    'villenave': "Villenave d'Ornon",
    'rheu': 'Le Rheu',
    'jouy': 'Jouy-en-Josas',
    'orsay': 'Orsay',
    'gif sur': 'Gif-sur-Yvette',
    'new york': 'New York',
    'london': 'London',
    'cambridge': 'Cambridge',
    'berlin': 'Berlin',
    'rome': 'Rome',
    'milan': 'Milan',
    'geneve': 'Genève',
    'zurich': 'Zurich',
    'barcelona': 'Barcelone',
    'cestas': 'Cestas',
    'guyancourt': 'Guyancourt',
    'montoldre': 'Montoldre',
    'limoges': 'Limoges',
    'reims': 'Reims',
    'dijon': 'Dijon',
    'caen': 'Caen',
    'clermont': 'Clermont-Ferrand',
    'tours': 'Tours',
    'aix': 'Aix-en-Provence',
}

def extract_city(address_list, country=None):
    """Extrait la ville à partir d'une adresse HAL."""
    if not address_list:
        return None
    addr = address_list[0] if isinstance(address_list, list) else address_list
    if not addr:
        return None
    addr_lower = addr.lower()
    
    # Recherche dans le dictionnaire de villes connues
    for key, val in CITY_MAP.items():
        if key in addr_lower:
            return val
    
    # Code postal français : 12345 NomVille
    m = re.search(r'\b\d{5}\s+([A-ZÀ-Ü][a-zA-ZÀ-ü\s\-\']+?)(?:\s*[,\n]|\s*$)', addr)
    if m:
        city = m.group(1).strip()
        city = re.sub(r'\s*[Cc]edex.*', '', city).strip()
        if len(city) > 2:
            return city
    
    # Dernier recours : dernier segment significatif
    parts = re.split(r'[\n,]', addr)
    for part in reversed(parts):
        part = part.strip()
        part = re.sub(r'\b\d{4,5}\b', '', part).strip()
        part = re.sub(r'[Cc]edex.*', '', part).strip()
        if len(part) > 3 and not part[0].isdigit():
            return part[:40]
    return None

print('✅ Fonctions utilitaires définies')

In [ ]:
def fetch_collection(collection_id, max_rows=2000):
    """Récupère toutes les publications d'une collection HAL."""
    url = HAL_BASE_URL.format(collection=collection_id)
    params = {
        'q': '*',
        'rows': max_rows,
        'fl': HAL_FIELDS,
        'wt': 'json'
    }
    print(f'📡 Récupération de {collection_id}...')
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    total = data['response']['numFound']
    docs = data['response']['docs']
    print(f'  ✅ {len(docs)}/{total} publications récupérées')
    return docs

def process_docs(docs):
    """Transforme les docs HAL bruts en records structurés."""
    records = []
    for doc in docs:
        addr_list = doc.get('labStructAddress_s', [])
        country = None
        if doc.get('labStructCountry_s'):
            country = doc['labStructCountry_s'][0]
        elif doc.get('instStructCountry_s'):
            country = doc['instStructCountry_s'][0]
        
        city = extract_city(addr_list, country)
        
        records.append({
            'halId': doc.get('halId_s'),
            'title': doc.get('title_s', [None])[0] if doc.get('title_s') else None,
            'docType': doc.get('docType_s'),
            'year': doc.get('publicationDateY_i'),
            'authors': doc.get('authFullName_s', []),
            'lab': doc.get('labStructName_s', [None])[0] if doc.get('labStructName_s') else None,
            'labAddress': addr_list[0] if addr_list else None,
            'city': city,
            'country': country,
            'institution': doc.get('instStructName_s', [None])[0] if doc.get('instStructName_s') else None,
            'domains': doc.get('domain_s', []),
            'keywords': doc.get('keyword_s', []),
            'journal': doc.get('journalTitle_s'),
            'language': doc.get('language_s', [None])[0] if doc.get('language_s') else None,
            'openAccess': doc.get('openAccess_bool', False),
            'doi': doc.get('doiId_s'),
        })
    return records

print('✅ Fonctions de traitement définies')

In [ ]:
def compute_stats(records):
    """Calcule les statistiques pour le dashboard."""
    unique_pubs = list({r['halId']: r for r in records}.values())
    
    years_count = {}
    doc_types = {}
    cities_count = {}
    countries_count = {}
    labs_count = {}
    authors_count = {}
    domains_count = {}
    
    for d in records:
        if d.get('year'): years_count[str(d['year'])] = years_count.get(str(d['year']), 0) + 1
        if d.get('docType'): doc_types[d['docType']] = doc_types.get(d['docType'], 0) + 1
        if d.get('city'): cities_count[d['city'].strip()] = cities_count.get(d['city'].strip(), 0) + 1
        c = (d.get('country') or 'fr').upper()
        countries_count[c] = countries_count.get(c, 0) + 1
        if d.get('lab'): labs_count[d['lab']] = labs_count.get(d['lab'], 0) + 1
        for dom in d.get('domains', []):
            lbl = dom.split('.')[-1].replace('-', ' ').title()
            domains_count[lbl] = domains_count.get(lbl, 0) + 1
    
    seen = set()
    for d in records:
        if d['halId'] not in seen:
            seen.add(d['halId'])
            for a in d.get('authors', []):
                authors_count[a] = authors_count.get(a, 0) + 1
    
    return {
        'total_pubs': len(unique_pubs),
        'total_entries': len(records),
        'years': sorted(years_count.items()),
        'doc_types': sorted(doc_types.items(), key=lambda x: -x[1]),
        'top_cities': sorted(cities_count.items(), key=lambda x: -x[1])[:15],
        'top_countries': sorted(countries_count.items(), key=lambda x: -x[1])[:20],
        'top_labs': sorted(labs_count.items(), key=lambda x: -x[1])[:15],
        'top_authors': sorted(authors_count.items(), key=lambda x: -x[1])[:20],
        'top_domains': sorted(domains_count.items(), key=lambda x: -x[1])[:12],
        'open_access': sum(1 for d in unique_pubs if d.get('openAccess')),
    }

print('✅ Fonction de stats définie')

In [ ]:
# === EXECUTION PRINCIPALE ===
for coll_id, config in COLLECTIONS.items():
    print(f'\n{'='*50}')
    print(f'Collection : {config["name"]} ({coll_id})')
    print('='*50)
    
    # Créer les dossiers
    out_dir = Path(config['output_dir'])
    out_dir.mkdir(parents=True, exist_ok=True)
    
    # Récupérer depuis HAL
    docs = fetch_collection(coll_id)
    
    # Traiter
    records = process_docs(docs)
    
    # Stats
    stats = compute_stats(records)
    
    # Sauvegarder
    with open(out_dir / 'publications.json', 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    with open(out_dir / 'stats.json', 'w', encoding='utf-8') as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)
    
    print(f'  📁 Sauvegardé dans {out_dir}/')
    print(f'  📊 {stats["total_pubs"]} publications uniques')
    print(f'  📅 Années : {[y for y,c in stats["years"]]}')
    cities_with_data = len([c for c,v in stats['top_cities']])
    print(f'  🏙️  {cities_with_data} villes identifiées')
    print(f'  🌍 {len(stats["top_countries"])} pays')
    print(f'  🔓 {stats["open_access"]} open access ({round(stats["open_access"]/max(stats["total_pubs"],1)*100)}%)')

print('\n✅ Toutes les données générées avec succès !')